# Notebook 1: Dataset Engineering & Hygiene Pipeline (Kaggle)
**Project**: Autonomous Text-to-SQL Generator Aligned via RLVR + GRPO  
**Hardware**: Kaggle CPU or GPU (Internet: ON)  
**Phase**: Phase 1 (Data Engineering, Gold Execution Cache, M-Schema Prompt Serialization)

---

In [ ]:
# Step 1: Install Pinned Dependencies for Kaggle
!pip install -q "sqlglot>=26.0.0" "datasets>=3.0.0" "pydantic>=2.8.0" "pyyaml>=6.0" "rich>=13.7.0" "huggingface_hub>=0.23.0"
print("Dependencies installed successfully.")

In [ ]:
# Step 2: Set up Working Directory Structure on Kaggle
import os
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIXTURES_DIR = DATA_DIR / "fixtures"

for d in [RAW_DIR, PROCESSED_DIR, FIXTURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Created directories under {DATA_DIR}")

In [ ]:
# Step 3: Download Spider 1.0 Dataset (Using High-Speed Verified Hugging Face Mirror)
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

spider_extracted = RAW_DIR / "spider"
spider_zip = RAW_DIR / "spider.zip"

# 1. Check if user already attached a Spider Kaggle dataset into /kaggle/input
kaggle_input_candidates = [
    Path("/kaggle/input/spider/spider"),
    Path("/kaggle/input/spider"),
    Path("/kaggle/input/spider-sqlite-databases"),
    Path("/kaggle/input/spider-nlp-dataset"),
]

found_in_input = False
for cand in kaggle_input_candidates:
    if (cand / "train_spider.json").exists() and (cand / "database").exists():
        print(f"Found Spider dataset in Kaggle input: {cand}")
        if spider_extracted.exists():
            shutil.rmtree(spider_extracted)
        shutil.copytree(cand, spider_extracted)
        found_in_input = True
        break

# 2. Check if already extracted in RAW_DIR
if not found_in_input:
    existing_train = list(RAW_DIR.glob("**/train_spider.json"))
    if existing_train:
        print("Spider dataset already present and extracted at:", existing_train[0].parent)
        found_in_input = True

# 3. Download from verified Hugging Face mirror
if not found_in_input:
    print("Downloading Spider 1.0 dataset (~99 MB) from verified Hugging Face mirror...")
    download_success = False

    # Method A: huggingface_hub (GuokLIU/db-upload is a model repo, repo_type='model')
    try:
        from huggingface_hub import hf_hub_download
        print("Attempting download via huggingface_hub API...")
        hf_path = hf_hub_download(repo_id="GuokLIU/db-upload", filename="spider.zip", repo_type="model")
        shutil.copy(hf_path, str(spider_zip))
        if spider_zip.exists() and spider_zip.stat().st_size > 50_000_000 and zipfile.is_zipfile(spider_zip):
            download_success = True
            print(f"Downloaded spider.zip via huggingface_hub ({spider_zip.stat().st_size / (1024*1024):.1f} MB).")
    except Exception as e:
        print(f"huggingface_hub notice: {e}")

    # Method B: Direct curl from verified URLs (without incorrect /datasets/ prefix)
    if not download_success:
        mirrors = [
            "https://huggingface.co/GuokLIU/db-upload/resolve/main/spider.zip",
            "https://huggingface.co/datasets/zmmjjkk/SqurveBridge-benchmarks/resolve/main/spider.zip",
        ]
        for url in mirrors:
            try:
                print(f"Fetching from {url} ...")
                if spider_zip.exists():
                    spider_zip.unlink()
                subprocess.run(["curl", "-L", "-k", "-f", "-o", str(spider_zip), url], check=True)
                if spider_zip.exists() and spider_zip.stat().st_size > 50_000_000 and zipfile.is_zipfile(spider_zip):
                    download_success = True
                    print(f"Direct download succeeded ({spider_zip.stat().st_size / (1024*1024):.1f} MB).")
                    break
            except Exception as err:
                print(f"Mirror failed: {err}")

    if not download_success or not zipfile.is_zipfile(spider_zip):
        raise RuntimeError("Failed to download valid spider.zip! Please ensure 'Internet' is toggled ON in Kaggle Notebook Settings.")

    print(f"Extracting spider.zip ({spider_zip.stat().st_size / (1024*1024):.1f} MB)... ")
    subprocess.run(["unzip", "-q", "-o", str(spider_zip), "-d", str(RAW_DIR)], check=True)

# Verify discovery of train_spider.json
found_files = list(RAW_DIR.glob("**/train_spider.json"))
if not found_files:
    raise RuntimeError(f"Extraction completed but train_spider.json not found under {RAW_DIR}!")
print("✅ Spider 1.0 dataset ready. Found train file at:", found_files[0])

In [ ]:
# Step 4: Schema Serializer (M-Schema with Column Types, PK, FK, and 3 Sample Values)
import sqlite3
from typing import Any, Dict, List, Optional, Set

class SchemaSerializer:
    def __init__(self, max_samples_per_col: int = 3, max_sample_chars: int = 50):
        self.max_samples_per_col = max_samples_per_col
        self.max_sample_chars = max_sample_chars

    def serialize(self, db_path: Path) -> Dict[str, Any]:
        uri = f"file:{db_path.resolve().as_posix()}?mode=ro"
        schema_info = {"tables": {}, "foreign_keys": []}
        with sqlite3.connect(uri, uri=True, timeout=2.0) as conn:
            conn.execute("PRAGMA query_only = ON")
            cur = conn.cursor()
            cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
            tables = [r[0] for r in cur.fetchall()]
            for tbl in tables:
                cur.execute(f"PRAGMA table_info('{tbl}');")
                cols = []
                pks = []
                for col in cur.fetchall():
                    cname, ctype, pk = col[1], col[2], bool(col[5])
                    if pk:
                        pks.append(cname)
                    samples = []
                    try:
                        cur.execute(f'SELECT DISTINCT "{cname}" FROM "{tbl}" WHERE "{cname}" IS NOT NULL LIMIT {self.max_samples_per_col};')
                        for s in cur.fetchall():
                            val_str = str(s[0]).strip()
                            if len(val_str) > self.max_sample_chars:
                                val_str = val_str[:self.max_sample_chars] + "..."
                            samples.append(val_str)
                    except Exception:
                        pass
                    cols.append({"name": cname, "type": ctype or "TEXT", "is_pk": pk, "samples": samples})
                cur.execute(f"PRAGMA foreign_key_list('{tbl}');")
                for fk in cur.fetchall():
                    schema_info["foreign_keys"].append({
                        "from_table": tbl, "from_column": fk[3],
                        "to_table": fk[2], "to_column": fk[4]
                    })
                schema_info["tables"][tbl] = {"columns": cols, "primary_keys": pks}
        return schema_info

    def to_compact_text(self, info: Dict[str, Any], external_knowledge: Optional[str] = None) -> str:
        lines = ["DATABASE SCHEMA:"]
        for tbl, tdata in sorted(info["tables"].items()):
            lines.append(f"  Table: {tbl}")
            for c in tdata["columns"]:
                pk_tag = ", PK" if c["is_pk"] else ""
                sample_tag = f" [examples: {', '.join(repr(s) for s in c['samples'])}]" if c['samples'] else ""
                lines.append(f"    - {c['name']} ({c['type']}{pk_tag}){sample_tag}")
        if info["foreign_keys"]:
            lines.append("  Foreign Keys:")
            for fk in info["foreign_keys"]:
                lines.append(f"    - {fk['from_table']}.{fk['from_column']} -> {fk['to_table']}.{fk['to_column']}")
        if external_knowledge:
            lines.append(f"\nEXTERNAL KNOWLEDGE:\n  {external_knowledge.strip()}")
        return "\n".join(lines)

serializer = SchemaSerializer()
print("Schema Serializer ready.")

In [ ]:
# Step 5: Process Spider Examples with Gold Execution Caching
import json
import time

SYSTEM_PROMPT = """You generate SQLite SELECT queries for the supplied database.
Rules:
- Use only the supplied schema and task-provided domain evidence.
- Return exactly one read-only SELECT-family SQL statement.
- Do not use DDL, DML, PRAGMA, ATTACH, DETACH or multiple statements.
- Do not invent tables or columns.
- Use the correct join path and predicates for the question.
- If the question cannot be answered from the schema, return a structured abstention.
- Output only the required response structure.

RESPONSE FORMAT:
<sql>SELECT ...</sql>"""

# Dynamically locate Spider files
train_candidates = list(RAW_DIR.glob("**/train_spider.json"))
dev_candidates = list(RAW_DIR.glob("**/dev.json"))
db_candidates = list(RAW_DIR.glob("**/database"))

assert train_candidates, f"train_spider.json not found under {RAW_DIR}!"
assert dev_candidates, f"dev.json not found under {RAW_DIR}!"
assert db_candidates, f"database folder not found under {RAW_DIR}!"

spider_train_path = train_candidates[0]
spider_dev_path = dev_candidates[0]
spider_db_root = db_candidates[0]

print(f"Using train file: {spider_train_path}")
print(f"Using dev file:   {spider_dev_path}")
print(f"Using DB root:    {spider_db_root}")

def process_spider_split(raw_json_path, split_name, max_samples=None):
    with open(raw_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if max_samples:
        data = data[:max_samples]
    
    records = []
    schema_cache = {}
    
    print(f"Processing {len(data)} examples for split: {split_name}...")
    for i, item in enumerate(data):
        db_id = item["db_id"]
        question = item["question"]
        gold_sql = item["query"]
        db_file = spider_db_root / db_id / f"{db_id}.sqlite"
        if not db_file.exists():
            continue
        
        if db_id not in schema_cache:
            s_info = serializer.serialize(db_file)
            s_text = serializer.to_compact_text(s_info)
            schema_cache[db_id] = (s_info, s_text)
        s_info, s_text = schema_cache[db_id]
        
        # Execute gold query with safe 1.0s limit
        gold_rows = None
        is_empty = False
        status = "success"
        try:
            uri = f"file:{db_file.resolve().as_posix()}?mode=ro"
            with sqlite3.connect(uri, uri=True, timeout=1.0) as conn:
                conn.execute("PRAGMA query_only = ON")
                c = conn.cursor()
                c.execute(gold_sql)
                gold_rows = c.fetchall()
                is_empty = (len(gold_rows) == 0)
                if is_empty:
                    status = "empty"
        except Exception:
            status = "error"
            gold_rows = None
        
        user_content = f"Database dialect: SQLite\n\n{s_text}\n\nQuestion: {question}"
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]
        
        records.append({
            "example_id": f"{split_name}_{i:06d}",
            "dataset": "spider",
            "split": split_name,
            "db_id": db_id,
            "question": question,
            "gold_sql": gold_sql,
            "gold_status": status,
            "gold_is_empty": is_empty,
            "gold_row_count": len(gold_rows) if gold_rows is not None else 0,
            "gold_result_json": json.dumps(gold_rows) if gold_rows is not None else "[]",
            "valid_tables": list(s_info["tables"].keys()),
            "prompt": prompt,
            "completion": f"<sql>{gold_sql}</sql>",
        })
        if (i + 1) % 500 == 0:
            print(f"  Progress: {i+1}/{len(data)} items processed...")
            
    print(f"✅ Completed {len(records)} verified records for {split_name}.")
    return records

# Process datasets
train_records = process_spider_split(spider_train_path, "train")
dev_records = process_spider_split(spider_dev_path, "dev")

In [ ]:
# Step 6: Save Processed Manifests to JSONL
def save_jsonl(records, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"Saved {len(records)} rows to {out_path} ({os.path.getsize(out_path)/1024:.1f} KB)")

train_out = PROCESSED_DIR / "train_manifest.jsonl"
dev_out = PROCESSED_DIR / "dev_manifest.jsonl"

save_jsonl(train_records, train_out)
save_jsonl(dev_records, dev_out)

empty_count = sum(1 for r in train_records if r["gold_is_empty"])
print(f"\nSummary: {len(train_records)} train records ({empty_count} empty results), {len(dev_records)} dev records.")

In [ ]:
# ==========================================================================
# FINAL CELL: MANDATORY ARTIFACT ZIPPER & DIRECT DOWNLOAD LINK
# Creates /kaggle/working/sql_grpo_processed_data.zip and displays a download link
# ==========================================================================
import subprocess
import os
from pathlib import Path
from IPython.display import FileLink, display

output_zip = WORKING_DIR / "sql_grpo_processed_data.zip"
if output_zip.exists():
    output_zip.unlink()

print(f"Creating {output_zip.name} ...")

# Use fast Linux zip command
cmd = f"cd {WORKING_DIR} && zip -q -r {output_zip} data/processed data/raw"
res = subprocess.run(cmd, shell=True)

# Fallback with python zipfile if needed
if not output_zip.exists() or output_zip.stat().st_size < 1000:
    import zipfile
    print("Using zipfile fallback...")
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for p in (WORKING_DIR / "data").rglob("*"):
            if p.is_file() and not p.name.endswith(".zip"):
                z.write(p, arcname=str(p.relative_to(WORKING_DIR)))

assert output_zip.exists(), "FATAL: Failed to create zip file!"
size_mb = output_zip.stat().st_size / (1024 * 1024)
print(f"\n✅ Successfully generated: {output_zip.name} ({size_mb:.2f} MB)")

# Display file in directory
print("\n--- Verification on Disk ---")
!ls -lh /kaggle/working/sql_grpo_processed_data.zip

print("\n👉 Click the link below to download directly in your browser:")
display(FileLink("sql_grpo_processed_data.zip"))
print("\n(Note: In the Kaggle right sidebar under 'Data -> Output', click the 🔄 refresh icon if not visible yet)")